In [1]:
import xarray as xr
import numpy as np
import xarray as xr
import pandas as pd
from sklearn.metrics import cohen_kappa_score


ds = xr.open_dataset("../../../NC/compare_region_basin_consistent_NODIR.nc")

df_map = pd.read_csv(
    "../../../CSV/gridset/RIJ_17regions.csv"
)

In [2]:
def build_region_masks_from_ij(ds, df_map):
    df_map = df_map.copy()
    df_map.columns = (
        df_map.columns
        .str.strip()
        .str.replace('\ufeff', '', regex=False)
    )

    required = {"Rall", "I", "J"}
    if not required.issubset(df_map.columns):
        raise ValueError(f"CSV columns must include {required}")

    base = ds["region_agri"].isel(time=0)
    ni, nj = base.shape

    region_masks = {}

    for region, g in df_map.groupby("Rall"):
        mask = xr.zeros_like(base, dtype=bool)

        # ★ 核心修正：1-based → 0-based
        ii = g["I"].to_numpy(dtype=int) - 1
        jj = g["J"].to_numpy(dtype=int) - 1

        # 安全检查（强烈建议保留）
        valid = (
            (ii >= 0) & (ii < ni) &
            (jj >= 0) & (jj < nj)
        )

        if not np.all(valid):
            print(f"[WARN] {region}: dropped {(~valid).sum()} grids out of bounds")

        mask.values[ii[valid], jj[valid]] = True
        region_masks[region] = mask

    return region_masks


In [3]:
def get_region_dominant_agri_ij(
    ds,
    region_mask,
    mode,
    time):
    da = ds[f"{mode}_agri"].sel(time=time)

    da = da.where(region_mask)
    da_filled = da.fillna(0)

    dom = xr.where(da_filled > 0, 1, 0)
    dom = dom.where(~np.isnan(da))

    return dom.values.flatten()


In [4]:
def calc_kappa(base, cur):
    mask = np.isfinite(base) & np.isfinite(cur)

    if mask.sum() == 0:
        return np.nan

    return round(
        cohen_kappa_score(
            base[mask].astype(int),
            cur[mask].astype(int)
        ),
        3
    )


In [5]:
regions = sorted(df_map["Rall"].unique().tolist())
regions.append("GLOBAL")

modes = ["region", "basin"]
years = [2005] + list(range(2010, 2101, 10))

region_masks = build_region_masks_from_ij(ds, df_map)

rows = []

for region in regions:
    print("Processing", region)

    for mode in modes:
        base_time = "2005-01-01"

        if region == "GLOBAL":
            base = xr.where(
                ds[f"{mode}_agri"].sel(time=base_time) > 0, 1, 0
            ).values.flatten()
        else:
            base = get_region_dominant_agri_ij(
                ds,
                region_masks[region],
                mode,
                base_time
            )

        for year in years[1:]:
            t = f"{year}-01-01"

            if region == "GLOBAL":
                cur = xr.where(
                    ds[f"{mode}_agri"].sel(time=t) > 0, 1, 0
                ).values.flatten()
            else:
                cur = get_region_dominant_agri_ij(
                    ds,
                    region_masks[region],
                    mode,
                    t
                )

            kappa = calc_kappa(base, cur)

            rows.append({
                "region": region,
                "mode": mode,
                "year": year,
                "kappa": kappa
            })


Processing BRA
Processing CAN
Processing CHN
Processing CIS
Processing IND
Processing JPN
Processing TUR
Processing USA
Processing XAF
Processing XE25
Processing XER
Processing XLM
Processing XME
Processing XNF
Processing XOC


d:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:409: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
d:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:730: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
d:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:409: UserWarning: A single label was found in 'y_true' and 'y_pred'. For the confusion matrix to have the correct shape, use the 'labels' parameter to pass all known labels.
  warnings.warn(
d:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:730: RuntimeWarning: invalid value encountered in scalar divide
  k = np.sum(w_mat * confusion) / np.sum(w_mat * expected)
d:\anaconda\Lib\site-packages\sklearn\metrics\_classification.py:409: UserWarning: A single label was found in '

Processing XSA
Processing XSE
Processing GLOBAL


In [6]:
df_out = pd.DataFrame(rows)

out_csv = "../../../CSV/kappa/kappa_agri_consistent_vs2005_NODIR.csv"
df_out.to_csv(out_csv, index=False, float_format="%.3f")

print("Saved:", out_csv)


Saved: ../../../CSV/kappa/kappa_agri_consistent_vs2005_NODIR.csv
